# Stage 4: Common re-evaluation, SVC calibration, thresholds and freeze

Thesis: *Explainable AI for Early Diabetes Prediction* (CDC BRFSS 2015). Config v1.0, master seed 42.

Implements Ch3 3.4.2 (E3.11), 3.4.3, 3.6.1 (E3.17), 3.6.3 (E3.20) and decisions **D7, D8, S12**, with S8 (UTF-8) and S10 (environment guard first).
Development data only: **the test partition is not loaded**.

1. Checks that the ten Stage 3 pipelines are the canonical files (SHA-256) and that their parameters equal the frozen configurations.
2. **Common 5-fold re-evaluation (D7):** each model's selected configuration is refitted on folds k' != k of the frozen development folds and scored on fold k. This gives fold AP (mean +/- SD) for every model and pooled out-of-fold (OOF) scores for all 202,944 development rows. For the eight models tuned on full data, the fold APs must reproduce Stage 3.
3. **Linear SVC calibration (3.4.3):** the frozen uncalibrated SVC is wrapped in 5-fold sigmoid calibration on the frozen development folds and refitted on the full development set. Its OOF probabilities come from the same procedure nested inside each outer fold (4 inner folds), so no row is scored by a calibrator that saw it.
4. **Thresholds (3.6.1, D8)** on the pooled OOF probabilities, with the rule *positive if probability >= t*: 0.50; the MCC-maximizing threshold (ties: higher sensitivity, then closer to 0.50, then lower); and the screening threshold, the highest t with sensitivity >= 0.80.
5. **Best model and contrasts (D7, E3.20):** best = highest mean re-evaluation AP; primary contrast = best vs LR; secondary = best vs the highest-ranked tree ensemble (RF, XGB, LGBM, CAT) other than itself.
6. **Freeze:** `frozen_models.json` (pipelines + hashes, thresholds, best model, contrasts). A rerun recomputes everything and must reproduce the frozen file exactly; it never overwrites it.

**Run:** put this notebook in `thesis_xai/` (same folder as Stage 3) and Run All. Expected time on the PC: about 10-20 minutes.

In [1]:
# Cell 1 - environment guard (S10): runs before numpy / scikit-learn are imported.
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:                                   # fallback machine only
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/thesis_xai"   # edit if your Drive folder differs
else:
    PROJECT_DIR = os.path.abspath(".")         # the PC: the folder that holds this notebook
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import env_setup
env_setup.ensure_pinned(PROJECT_DIR, install=IN_COLAB)

All pinned packages present at the pinned versions.


{}

In [2]:
# Cell 2 - configuration
import ast
import hashlib
import json
import platform
import time
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, roc_auc_score

import thesis_common as tc

MACHINE = "colab" if IN_COLAB else "pc"
STAGE = "stage4_reeval_freeze.ipynb"
MODELS = list(tc.MODELS)                     # all ten; do not change for the thesis run
TREE_ENSEMBLES = ["RF", "XGB", "LGBM", "CAT"]  # E3.20 secondary-contrast pool
SCREEN_SENS = 0.80                           # D8
REPRO_TOL = 1e-4                             # Stage 3 fold-AP reproduction: stop above this
REPRO_WARN = 1e-9                            # record (not stop) above this

# Canonical Stage 3 pipelines (stage3_summary.csv, PC run 2026-10-12; registry S3).
CANON_STAGE3_SHA = {
    "GNB": "70939cdb6e8259504f1991ec503aa1d6ba88be995f0574aa750cf3d81d1143e1",
    "DT": "3146bde7cdbf7ea2c4386a368c742b6d3eaf6633b7d417b8bcf9c2f4fe61eb72",
    "LR": "2deb4902b2c4030c3d9a5cfcdc8474d9a8754462fd6378e65cc6711ca1c90966",
    "SVC": "2f21d97962d0291f6f24ffeb0cbf511297b333f48b164d9a616a714a81b06336",
    "KNN": "3587dadbec9d69a57a3617a1679da602eed01471a07dbc03655e8f847bcdada4",
    "XGB": "f397dc6db4095323b18c007b9cd8f22668c6b89c2bce53ae760a9460bdc7b062",
    "LGBM": "4b5d7646db5457245c4c8f8a7f6a360cfb2d8f8de50176e92aeffc18bf671997",
    "MLP": "e58354518e9b8b13042f9cdde8b8576019a1b7bf1c5fd247cf8a8d2cf58c744f",
    "CAT": "20a890f1b8fe43ffe200e0f75d35addd1c59d9c9add27c47e577667d89d92472",
    "RF": "3be59116e250dd553b631511bd8392087d8144997892f4569bdae68253f0ea99",
}

EXPECTED = {"dev_rows": 202944, "dev_pos": 28277, "dev_fold_sizes": [40589, 40589, 40589, 40589, 40588],
            "sub_rows": 50000, "sub_pos": 6965, "sub_fold_sizes": [10000] * 5, "n_raw": 253680}

S3 = os.path.join(PROJECT_DIR, "stage3_outputs", "models")
OUT = os.path.join(PROJECT_DIR, "stage4_outputs")
OOF_DIR = os.path.join(OUT, "oof")
MODEL_DIR = os.path.join(OUT, "models")
for d in (OUT, OOF_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

assert tc.MODEL_SEED == 22980462 and tc.N_FOLDS == 5
print(f"machine={MACHINE}  python={platform.python_version()}  scikit-learn={sklearn.__version__}  cpu_count={os.cpu_count()}")

machine=pc  python=3.13.0  scikit-learn=1.6.1  cpu_count=12


In [3]:
# Cell 3 - development data and frozen folds (the test rows are dropped at once and never used).
ROW_ID_NAMES = ["row_id", "row", "idx", "index"]


def row_ids(df, name):
    col = next((c for c in ROW_ID_NAMES if c in df.columns), None)
    if col is None or "fold" not in df.columns:
        raise SystemExit(f"{name}: expected a row-id column ({ROW_ID_NAMES}) and 'fold'; found {list(df.columns)}")
    r = df[col].to_numpy()
    if not (np.issubdtype(r.dtype, np.integer) and r.min() >= 0 and r.max() < EXPECTED["n_raw"]
            and len(np.unique(r)) == len(r)):
        raise SystemExit(f"{name}: column '{col}' is not a set of unique raw-row positions")
    return r


X_all, y_all = tc.load_data(PROJECT_DIR)                 # SHA-256 checked
idx = tc.load_indices(PROJECT_DIR)                        # every index file checked against MANIFEST.csv
dev_folds = idx["dev_folds.csv"]
del idx
dev_rows = row_ids(dev_folds, "dev_folds.csv")
X_dev = X_all.iloc[dev_rows].reset_index(drop=True)
y_dev = y_all[dev_rows]
del X_all, y_all
fold_id = dev_folds["fold"].to_numpy()
cv_dev = tc.fold_splits(dev_folds)

checks = {"development rows": (len(X_dev), EXPECTED["dev_rows"]),
          "development positives": (int(y_dev.sum()), EXPECTED["dev_pos"]),
          "development fold sizes": ([len(v) for _, v in cv_dev], EXPECTED["dev_fold_sizes"])}
for k, (got, want) in checks.items():
    print(f"{'PASS' if got == want else 'FAIL'}  {k}: {got}")
if any(got != want for got, want in checks.values()):
    raise SystemExit("Development data checks failed.")

PASS  development rows: 202944
PASS  development positives: 28277
PASS  development fold sizes: [40589, 40589, 40589, 40589, 40588]


In [4]:
# Cell 4 - the Stage 3 results must be the canonical ones.
def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def write_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


S3INFO = {}
rows = []
for m in MODELS:
    pk = os.path.join(S3, m, f"pipeline_{m}.joblib")
    b = read_json(os.path.join(S3, m, f"best_params_{m}.json"))
    params = {k: ast.literal_eval(v) for k, v in b["params_repr"].items()}
    sha = tc.sha256_file(pk)
    ok_sha = sha == CANON_STAGE3_SHA[m]
    ok_par = params == tc.sampled_configs(m)[b["config_id"]]
    S3INFO[m] = {"params": params, "config_id": b["config_id"], "fold_ap": b["fold_ap"], "pipeline": pk,
                 "sha256": sha, "tuned_on": b["tuned_on"]}
    rows.append({"model": m, "config_id": b["config_id"], "pipeline_sha_ok": ok_sha, "params_equal_frozen": ok_par})
s3chk = pd.DataFrame(rows)
print(s3chk.to_string(index=False))
if not (s3chk.pipeline_sha_ok.all() and s3chk.params_equal_frozen.all()):
    raise SystemExit("A Stage 3 pipeline or parameter file is not the canonical one. Restore stage3_outputs.")

model  config_id  pipeline_sha_ok  params_equal_frozen
   LR         20             True                 True
   DT          3             True                 True
   RF          6             True                 True
  SVC         20             True                 True
  KNN         16             True                 True
  GNB         16             True                 True
  XGB          2             True                 True
 LGBM         19             True                 True
  CAT          7             True                 True
  MLP         21             True                 True


In [5]:
# Cell 5 - re-evaluation functions
def calibrated_svc(params, inner_splits):
    # 5-fold (final) or 4-fold (nested) sigmoid calibration of the frozen, uncalibrated SVC pipeline (3.4.3)
    base = tc.make_pipeline("SVC").set_params(**params)
    return CalibratedClassifierCV(base, method="sigmoid", cv=inner_splits, ensemble=True)


def inner_splits(train_pos):
    # inner CV for nested calibration: the remaining four frozen development folds
    f = fold_id[train_pos]
    return [(np.flatnonzero(f != k), np.flatnonzero(f == k)) for k in np.unique(f)]


def oof_path(m):
    return os.path.join(OOF_DIR, f"oof_{m}.csv")


def reevaluate(m):
    # Pooled OOF scores and per-fold metrics for model m on the five full development folds.
    if os.path.exists(oof_path(m) + ".done"):
        df = pd.read_csv(oof_path(m), float_precision="round_trip")
        print(f"{m}: OOF already computed - loaded")
        return df
    p = S3INFO[m]["params"]
    score = np.full(len(y_dev), np.nan)
    raw = np.full(len(y_dev), np.nan) if m == "SVC" else None
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        for k, (tr, va) in enumerate(cv_dev):
            pipe = tc.make_pipeline(m, n_jobs=-1).set_params(**p).fit(X_dev.iloc[tr], y_dev[tr])
            if m == "RF":   # RF sums tree probabilities across threads in varying order; predict single-threaded so scores are bit-stable (S12)
                pipe.set_params(clf__n_jobs=1)
            s = tc.positive_score(pipe, X_dev.iloc[va])            # SVC: decision function
            if m == "SVC":
                raw[va] = s
                cal = calibrated_svc(p, inner_splits(tr)).fit(X_dev.iloc[tr], y_dev[tr])
                s = cal.predict_proba(X_dev.iloc[va])[:, 1]
            score[va] = s
    df = pd.DataFrame({"row_id": dev_rows, "fold": fold_id, "y": y_dev, "score": score})
    if raw is not None:
        df["svc_decision_uncalibrated"] = raw
    if not np.isfinite(df["score"]).all():
        raise SystemExit(f"{m}: non-finite OOF scores")
    df.to_csv(oof_path(m), index=False, encoding="utf-8", float_format="%.17g", lineterminator="\n")
    wl = sorted({f"{x.category.__name__}: {str(x.message)[:200]}" for x in w})
    write_json({"model": m, "machine": MACHINE, "seconds": round(time.perf_counter() - t0, 1), "warnings": wl,
                "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, oof_path(m) + ".done")
    back = pd.read_csv(oof_path(m), float_precision="round_trip")   # exact float round trip (pandas' default parser is not)
    if not np.array_equal(back["score"].to_numpy(), score):
        raise SystemExit(f"{m}: saved OOF scores do not round-trip exactly")
    print(f"{m}: OOF done in {time.perf_counter() - t0:.0f} s, warnings {len(wl)}")
    return back


def fold_metrics(df, col="score"):
    ap = [average_precision_score(df.y[df.fold == k], df[col][df.fold == k]) for k in range(tc.N_FOLDS)]
    auc = [roc_auc_score(df.y[df.fold == k], df[col][df.fold == k]) for k in range(tc.N_FOLDS)]
    return np.array(ap), np.array(auc)

In [6]:
# Cell 6 - run the re-evaluation (checkpointed per model; a rerun loads finished models).
OOF = {m: reevaluate(m) for m in ["GNB", "DT", "LR", "SVC", "KNN", "XGB", "LGBM", "MLP", "CAT", "RF"] if m in MODELS}

GNB: OOF done in 1 s, warnings 0
DT: OOF done in 4 s, warnings 0
LR: OOF done in 7 s, warnings 0
SVC: OOF done in 22 s, warnings 0
KNN: OOF done in 168 s, warnings 0
XGB: OOF done in 25 s, warnings 0
LGBM: OOF done in 21 s, warnings 3
MLP: OOF done in 52 s, warnings 0
CAT: OOF done in 65 s, warnings 0
RF: OOF done in 39 s, warnings 0


In [7]:
# Cell 7 - D7 table and Stage 3 reproduction check
rows, repro = [], []
for m in MODELS:
    ap, auc = fold_metrics(OOF[m])
    r = {"model": m, "tuned_on": "50k subsample" if m in tc.SUBSAMPLE_TUNED else "full development",
         "mean_ap": ap.mean(), "sd_ap": ap.std(ddof=0), **{f"ap_fold{k}": ap[k] for k in range(5)},
         "mean_roc_auc": auc.mean(), "sd_roc_auc": auc.std(ddof=0), "pooled_ap": average_precision_score(OOF[m].y, OOF[m].score)}
    if m == "SVC":   # the uncalibrated decision function must reproduce Stage 3
        ap_raw, _ = fold_metrics(OOF[m], "svc_decision_uncalibrated")
        r["mean_ap_uncalibrated"] = ap_raw.mean()
    rows.append(r)
    if m not in tc.SUBSAMPLE_TUNED:
        a = ap_raw if m == "SVC" else ap
        d = float(np.max(np.abs(a - np.array(S3INFO[m]["fold_ap"]))))
        repro.append({"model": m, "max_abs_diff_vs_stage3": d,
                      "status": "identical" if d <= REPRO_WARN else ("tiny difference" if d <= REPRO_TOL else "MISMATCH")})
reeval = pd.DataFrame(rows).sort_values("mean_ap", ascending=False, kind="mergesort").reset_index(drop=True)  # exact ties keep tc.MODELS order
reeval.insert(1, "rank", np.arange(1, len(reeval) + 1))
reeval.to_csv(os.path.join(OUT, "reeval_5fold_dev.csv"), index=False, encoding="utf-8", float_format="%.10g")
repro = pd.DataFrame(repro)
repro.to_csv(os.path.join(OUT, "stage3_reproduction_check.csv"), index=False, encoding="utf-8")
pd.set_option("display.width", 200)
print(reeval[["rank", "model", "tuned_on", "mean_ap", "sd_ap", "mean_roc_auc", "sd_roc_auc"]].round(4).to_string(index=False))
print("\n" + repro.to_string(index=False))
if (repro.status == "MISMATCH").any():
    raise SystemExit("Re-evaluation does not reproduce Stage 3 fold AP for a model tuned on full data. Stop and report.")

 rank model         tuned_on  mean_ap  sd_ap  mean_roc_auc  sd_roc_auc
    1   CAT full development   0.4374 0.0077        0.8315      0.0032
    2   XGB full development   0.4363 0.0067        0.8309      0.0028
    3  LGBM full development   0.4363 0.0072        0.8309      0.0029
    4   MLP    50k subsample   0.4306 0.0071        0.8287      0.0035
    5    RF full development   0.4299 0.0071        0.8271      0.0032
    6   KNN    50k subsample   0.4209 0.0057        0.8234      0.0036
    7    DT full development   0.4066 0.0081        0.8148      0.0037
    8    LR full development   0.4058 0.0042        0.8224      0.0032
    9   SVC full development   0.4054 0.0041        0.8227      0.0030
   10   GNB full development   0.3505 0.0042        0.7937      0.0020

model  max_abs_diff_vs_stage3    status
   LR                     0.0 identical
   DT                     0.0 identical
   RF                     0.0 identical
  SVC                     0.0 identical
  GNB             

In [8]:
# Cell 8 - thresholds on pooled OOF probabilities (3.6.1 / E3.17 / D8). Rule: positive if p >= t.
def threshold_table(y, s):
    # every candidate threshold = a distinct pooled OOF value; counts for 'p >= t'
    order = np.argsort(-s, kind="mergesort")
    ss, yy = s[order], y[order]
    last = np.r_[np.flatnonzero(np.diff(ss) != 0), len(ss) - 1]
    tp = np.cumsum(yy)[last].astype(float)
    fp = np.cumsum(1 - yy)[last].astype(float)
    P, N = float(y.sum()), float(len(y) - y.sum())
    fn, tn = P - tp, N - fp
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0.0)
    return pd.DataFrame({"t": ss[last], "tp": tp, "fp": fp, "fn": fn, "tn": tn, "sens": tp / P, "mcc": mcc})


def pick_mcc(tt):
    # max MCC; ties: higher sensitivity, then closer to 0.50, then lower threshold
    i = np.lexsort((tt.t.to_numpy(), np.abs(tt.t.to_numpy() - 0.5), -tt.sens.to_numpy(), -tt.mcc.to_numpy()))[0]
    return float(tt.t.iloc[i]), int((tt.mcc == tt.mcc.iloc[i]).sum())


def pick_screen(tt):
    ok = tt[tt.sens >= SCREEN_SENS]
    return float(ok.t.max())


def cls_metrics(y, s, t):
    p = s >= t
    tp, fp = int((p & (y == 1)).sum()), int((p & (y == 0)).sum())
    fn, tn = int((~p & (y == 1)).sum()), int((~p & (y == 0)).sum())
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    prec = tp / (tp + fp) if tp + fp else float("nan")
    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn, "sensitivity": sens, "specificity": spec, "precision": prec,
            "f1": 2 * tp / (2 * tp + fp + fn), "mcc": (tp * tn - fp * fn) / den if den else 0.0,
            "balanced_accuracy": (sens + spec) / 2, "accuracy": (tp + tn) / len(y), "predicted_positive_rate": (tp + fp) / len(y)}


THR, rows = {}, []
for m in MODELS:
    y, s = OOF[m].y.to_numpy(), OOF[m].score.to_numpy()
    if s.min() < 0 or s.max() > 1:
        raise SystemExit(f"{m}: OOF scores are not probabilities")
    tt = threshold_table(y, s)
    t_mcc, n_tie = pick_mcc(tt)
    THR[m] = {"t050": 0.5, "t_mcc": t_mcc, "t_screen": pick_screen(tt), "mcc_ties": n_tie, "n_candidates": len(tt)}
    for name, t in [("0.50", 0.5), ("MCC-max", t_mcc), ("sens>=0.80", THR[m]["t_screen"])]:
        rows.append({"model": m, "operating_point": name, "threshold": t, **cls_metrics(y, s, t)})
thr_dev = pd.DataFrame(rows)
thr_dev.to_csv(os.path.join(OUT, "thresholds_dev_oof_metrics.csv"), index=False, encoding="utf-8", float_format="%.10g")
print(thr_dev[["model", "operating_point", "threshold", "sensitivity", "specificity", "precision", "mcc",
               "predicted_positive_rate"]].round(4).to_string(index=False))

model operating_point  threshold  sensitivity  specificity  precision    mcc  predicted_positive_rate
   LR            0.50     0.5000       0.1577       0.9780     0.5374 0.2373                   0.0409
   LR         MCC-max     0.1861       0.6667       0.7996     0.3501 0.3657                   0.2653
   LR      sens>=0.80     0.1253       0.8000       0.6943     0.2976 0.3537                   0.3745
   DT            0.50     0.5000       0.7843       0.6938     0.2931 0.3424                   0.3728
   DT         MCC-max     0.6235       0.6427       0.8099     0.3538 0.3605                   0.2532
   DT      sens>=0.80     0.4847       0.8000       0.6799     0.2880 0.3412                   0.3870
   RF            0.50     0.5000       0.1214       0.9870     0.6020 0.2272                   0.0281
   RF         MCC-max     0.2286       0.6256       0.8325     0.3768 0.3762                   0.2313
   RF      sens>=0.80     0.1408       0.8000       0.6998     0.3014 0.3585      

In [9]:
# Cell 9 - final calibrated Linear SVC: 5-fold sigmoid calibration on the frozen development folds, fitted on all development rows.
svc_path = os.path.join(MODEL_DIR, "pipeline_SVC_calibrated.joblib")
if not os.path.exists(svc_path):
    t0 = time.perf_counter()
    svc_cal = calibrated_svc(S3INFO["SVC"]["params"], cv_dev).fit(X_dev, y_dev)
    svc_fit_s = time.perf_counter() - t0
    joblib.dump(svc_cal, svc_path + ".tmp", compress=3)
    os.replace(svc_path + ".tmp", svc_path)
    write_json({"fit_seconds": round(svc_fit_s, 2), "machine": MACHINE}, svc_path + ".json")
svc_cal = joblib.load(svc_path)
assert len(svc_cal.calibrated_classifiers_) == 5 and svc_cal.method == "sigmoid"
print("calibrated SVC:", os.path.basename(svc_path), tc.sha256_file(svc_path)[:16], "| 5 sigmoid calibrators")

calibrated SVC: pipeline_SVC_calibrated.joblib c26321360c405aa3 | 5 sigmoid calibrators


In [10]:
# Cell 10 - best model, contrasts and freeze (before any test data is used)
order = reeval.model.tolist()                     # sorted by mean re-evaluation AP, highest first
best = order[0]
secondary = next((m for m in order if m in TREE_ENSEMBLES and m != best), None)
primary = None if best == "LR" else [best, "LR"]       # if LR itself is best, the primary contrast does not exist
contrasts = {"primary": primary, "secondary": [best, secondary]}
if primary is None:
    print("NOTE: Logistic Regression is the best model; the primary contrast (best vs LR) is not applicable.")
print(f"Best model (highest mean development AP, D7): {best}")
PRIMARY_TXT = f"{best} vs LR" if primary else "not applicable (LR is best)"
print(f"Primary contrast: {PRIMARY_TXT} | secondary contrast: {best} vs {secondary}")

frozen = {"stage": STAGE, "config_version": tc.CONFIG_VERSION, "threshold_rule": "positive if probability >= t",
          "best_model": best, "contrasts": contrasts,
          "best_rule": "highest mean 5-fold re-evaluation AP on development folds (D7)",
          "models": {}}
for m in MODELS:
    if m == "SVC":
        pfile, sha = os.path.relpath(svc_path, PROJECT_DIR), tc.sha256_file(svc_path)
    else:
        pfile, sha = os.path.relpath(S3INFO[m]["pipeline"], PROJECT_DIR), S3INFO[m]["sha256"]
    r = reeval.set_index("model").loc[m]
    frozen["models"][m] = {"pipeline": pfile.replace(os.sep, "/"), "pipeline_sha256": sha,
                           "score": "calibrated probability (sigmoid, 5-fold)" if m == "SVC" else "predict_proba[:, 1]",
                           "config_id": S3INFO[m]["config_id"],
                           "params_repr": {k: repr(v) for k, v in S3INFO[m]["params"].items()},
                           "dev_mean_ap": round(float(r.mean_ap), 10), "dev_sd_ap": round(float(r.sd_ap), 10),
                           "threshold_050": 0.5, "threshold_mcc": THR[m]["t_mcc"], "threshold_screen": THR[m]["t_screen"],
                           "oof_file": os.path.relpath(oof_path(m), PROJECT_DIR).replace(os.sep, "/"),
                           "oof_sha256": tc.sha256_file(oof_path(m))}
blob = json.dumps(frozen, indent=2, ensure_ascii=False, sort_keys=True).encode("utf-8")
fz = os.path.join(OUT, "frozen_models.json")
if os.path.exists(fz):
    with open(fz, "rb") as f:
        same = f.read() == blob
    if not same:
        raise SystemExit("frozen_models.json exists and this run differs from it. The freeze is never overwritten; report this.")
    print("frozen_models.json: verified identical to the existing freeze")
else:
    with open(fz, "wb") as f:
        f.write(blob)
    print("frozen_models.json: created")
print("freeze sha256:", hashlib.sha256(blob).hexdigest())

Best model (highest mean development AP, D7): CAT
Primary contrast: CAT vs LR | secondary contrast: CAT vs XGB
frozen_models.json: created
freeze sha256: 1dd5bf06708d148399675d9fc657857df1c86f015cbd83c24e79e8754b48c0d8


In [11]:
# Cell 11 - report and environment log
env = tc.environment(MACHINE)
write_json({**env, "stage": STAGE, "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "pins_differ": {k: list(v) for k, v in tc.check_pins(PROJECT_DIR).items()},
            "oof_seconds": {m: read_json(oof_path(m) + ".done")["seconds"] for m in MODELS}},
           os.path.join(OUT, f"env_log_stage4_{MACHINE}.json"))
L = ["# Stage 4 report: re-evaluation, calibration, thresholds and freeze", "",
     f"Generated by `{STAGE}` (config {tc.CONFIG_VERSION}) on {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC, machine {MACHINE}, "
     f"Python {env['python']}, scikit-learn {env['packages']['scikit-learn']}. Development data only; the test partition was not loaded.", "",
     "## Common 5-fold re-evaluation (D7)", "",
     "| Rank | Model | Tuned on | Mean AP (SD) | Mean ROC-AUC (SD) |", "| --- | --- | --- | --- | --- |"]
for r in reeval.itertuples():
    L.append(f"| {r.rank} | {r.model} | {r.tuned_on} | {r.mean_ap:.4f} ({r.sd_ap:.4f}) | {r.mean_roc_auc:.4f} ({r.sd_roc_auc:.4f}) |")
L += ["", "SVC: calibrated OOF probabilities (nested sigmoid calibration). Fold SD uses ddof = 0, as in Stage 3. "
      "For the eight models tuned on full data these values reproduce the Stage 3 selection scores, so they carry the "
      "selection optimism of choosing the best of 30 configurations; the test set gives the unbiased estimate.", "",
      "Stage 3 reproduction: " + ", ".join(f"{r.model} {r.status}" for r in repro.itertuples()) + ".", "",
      f"**Best model:** {best}. **Primary contrast:** {PRIMARY_TXT}. **Secondary contrast:** {best} vs {secondary}.", "",
      "## Frozen thresholds (pooled OOF probabilities; positive if p >= t)", "",
      "| Model | MCC-max t | Sens / Spec / MCC at t | Screening t | Sens / Spec at screening t |", "| --- | --- | --- | --- | --- |"]
for m in MODELS:
    a = thr_dev[(thr_dev.model == m) & (thr_dev.operating_point == "MCC-max")].iloc[0]
    b = thr_dev[(thr_dev.model == m) & (thr_dev.operating_point == "sens>=0.80")].iloc[0]
    L.append(f"| {m} | {THR[m]['t_mcc']:.6g} | {a.sensitivity:.3f} / {a.specificity:.3f} / {a.mcc:.3f} "
             f"| {THR[m]['t_screen']:.6g} | {b.sensitivity:.3f} / {b.specificity:.3f} |")
L += ["", "Development OOF metrics at the thresholds are descriptive (they are where the thresholds were chosen); "
      "reported threshold performance comes from the test set (Stage 5).", "",
      f"Freeze: `stage4_outputs/frozen_models.json`, sha256 {hashlib.sha256(blob).hexdigest()}."]
with open(os.path.join(OUT, "stage4_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(L) + "\n")
print("\n".join(L))
print("\nSend back: stage4_outputs/stage4_report.md, reeval_5fold_dev.csv, stage3_reproduction_check.csv, "
      f"thresholds_dev_oof_metrics.csv, frozen_models.json, env_log_stage4_{MACHINE}.json, oof/*.done "
      "(not the oof_*.csv or .joblib files).")

# Stage 4 report: re-evaluation, calibration, thresholds and freeze

Generated by `stage4_reeval_freeze.ipynb` (config v1.0) on 2026-10-13 16:23 UTC, machine pc, Python 3.13.0, scikit-learn 1.6.1. Development data only; the test partition was not loaded.

## Common 5-fold re-evaluation (D7)

| Rank | Model | Tuned on | Mean AP (SD) | Mean ROC-AUC (SD) |
| --- | --- | --- | --- | --- |
| 1 | CAT | full development | 0.4374 (0.0077) | 0.8315 (0.0032) |
| 2 | XGB | full development | 0.4363 (0.0067) | 0.8309 (0.0028) |
| 3 | LGBM | full development | 0.4363 (0.0072) | 0.8309 (0.0029) |
| 4 | MLP | 50k subsample | 0.4306 (0.0071) | 0.8287 (0.0035) |
| 5 | RF | full development | 0.4299 (0.0071) | 0.8271 (0.0032) |
| 6 | KNN | 50k subsample | 0.4209 (0.0057) | 0.8234 (0.0036) |
| 7 | DT | full development | 0.4066 (0.0081) | 0.8148 (0.0037) |
| 8 | LR | full development | 0.4058 (0.0042) | 0.8224 (0.0032) |
| 9 | SVC | full development | 0.4054 (0.0041) | 0.8227 (0.0030) |
| 10 | GNB | full